# Q5 — Mini Try-On Web Demo

Wraps **Q1** (VLM attribute extraction), **Q2** (human parsing / agnostic + garment masks),
**Q3** (CatVTON virtual try-on inference) and **Q4** (automated quality evaluation) into a single
**Gradio** web app.

**Guardrails implemented:**
- ❌ **Reject** if no person is detected in the uploaded person image (SCHP parsing finds only
  background, confirmed by the Q1 VLM `pose_category == "unknown"` no-person guard).
- ⚠️ **Warn** (but still process) if the detected pose is `seated` or `side` — these poses are
  known to produce less reliable try-on geometry.
- ⏱️ **Estimated processing time** is shown before the run starts and the actual measured time
  is shown after it finishes.

**How to run:**
1. Run every cell top‑to‑bottom (Runtime → Run all). Use a **T4 GPU** runtime.
2. The last cell launches a Gradio app with a public `share=True` link — open it, upload a
   person + garment image, and click **Run Try-On**.
3. The final cell also demonstrates the guardrails directly on
   `edge_cases/no_person.jpg`, `edge_cases/person_seated.jpg`, `edge_cases/person_side_pose.jpg`
   — record your demo video using that cell (or by uploading the same 3 images in the Gradio UI).

> This notebook assumes the same Google Drive project layout used in Q1–Q4:
> `/content/drive/MyDrive/XIPL_SDE_Assessment/{Q1,Q2,Q3,Q4}/...`


## 1. Mount Drive & project paths

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_DIR = "/content/drive/MyDrive/XIPL_SDE_Assessment"
Q5_DIR = os.path.join(PROJECT_DIR, "Q5")
os.makedirs(Q5_DIR, exist_ok=True)

print("Project dir:", PROJECT_DIR)
print("Q5 dir     :", Q5_DIR)
print("Exists     :", os.path.exists(PROJECT_DIR))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project dir: /content/drive/MyDrive/XIPL_SDE_Assessment
Q5 dir     : /content/drive/MyDrive/XIPL_SDE_Assessment/Q5
Exists     : True


## 2. Install dependencies

Same stack used across Q1–Q4, plus `gradio` for the web UI.

In [2]:
# ------------------------------------------------------------
# Core ML / CV stack (matches Q1-Q4 pinned versions)
# ------------------------------------------------------------
!pip uninstall -y transformers tokenizers -q

!pip install -q \
    "transformers==4.46.3" \
    "tokenizers<0.21" \
    "sentencepiece==0.1.99" \
    "accelerate==0.31.0" \
    "bitsandbytes>=0.46.1" \
    "peft==0.12.0" \
    "diffusers==0.30.3" \
    "huggingface_hub>=0.25.0,<1.0" \
    "safetensors" \
    "omegaconf==2.3.0" \
    "fvcore==0.1.5.post20221221" \
    iopath \
    "pycocotools==2.0.8" \
    "av==12.3.0" \
    pydantic \
    opencv-python \
    scikit-image \
    open_clip_torch \
    rembg \
    onnxruntime \
    pandas \
    matplotlib \
    tqdm \
    gradio

!pip uninstall -y Pillow -q
!pip install --no-cache-dir --force-reinstall -q "Pillow==10.4.0"

print("\n✅ Dependencies installed. If this is the FIRST install in this runtime,")
print("   restart the runtime now (Runtime -> Restart session) then re-run from Cell 1,")
print("   skipping the pip installs the second time around.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 66.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 5.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.4/309.4 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 89.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 443.5/443.5 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.7/34.7 MB 28.4 MB/s eta 0:00:00
  

## 3. Clone CatVTON (Q3 model repo)

In [2]:
import os

CATVTON_DIR = "/content/CatVTON"

if not os.path.exists(CATVTON_DIR):
    !git clone https://github.com/Zheng-Chong/CatVTON.git /content/CatVTON
else:
    print("CatVTON repository already exists.")

print("Exists:", os.path.exists(CATVTON_DIR))


Cloning into '/content/CatVTON'...
remote: Enumerating objects: 1358, done.
remote: Counting objects: 100% (309/309), done.
remote: Compressing objects: 100% (181/181), done.
remote: Total 1358 (delta 169), reused 128 (delta 128), pack-reused 1049 (from 2)
Receiving objects: 100% (1358/1358), 16.73 MiB | 14.46 MiB/s, done.
Resolving deltas: 100% (462/462), done.
Exists: True


## 4. Load models

All four models used across Q1–Q4 are loaded once at startup:
- **SCHP** (`pirocheto/schp-atr-18`) — human parsing (Q2)
- **CatVTON** (`booksforcharlie/stable-diffusion-inpainting` + `zhengchong/CatVTON`, vitonhd) — try-on generation (Q3)
- **OpenCLIP** (`ViT-B-32`, `laion2b_s34b_b79k`) — garment fidelity (Q4)
- **OpenCV Haar cascade** — face detection for identity preservation (Q4)
- **MiniCPM-V-2_6-int4** — VLM for Q1 attribute extraction *and* the Q4 VLM-as-judge

On a T4 (16GB) this fits, but it is tight — if you hit an OOM, restart the runtime and only run this cell once (don't re-run it, which would load duplicate copies).

In [4]:
import torch, gc
import numpy as np
import cv2
from PIL import Image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# 4a. SCHP human parsing model (Q2)
# ------------------------------------------------------------
from huggingface_hub import snapshot_download

# Download the full repo (weights + code files) to a local folder
schp_local_dir = snapshot_download(
    repo_id="pirocheto/schp-atr-18",
    local_dir="/content/schp-atr-18",
)

import sys
if schp_local_dir not in sys.path:
    sys.path.insert(0, schp_local_dir)

from transformers import AutoImageProcessor, AutoModelForSemanticSegmentation

schp_processor = AutoImageProcessor.from_pretrained(
    schp_local_dir, trust_remote_code=True
)
schp_model = AutoModelForSemanticSegmentation.from_pretrained(
    schp_local_dir, trust_remote_code=True
).to(DEVICE).eval()

print("✅ SCHP loaded")

# ------------------------------------------------------------
# 4b. CatVTON pipeline (Q3)
# ------------------------------------------------------------
import sys
if CATVTON_DIR not in sys.path:
    sys.path.insert(0, CATVTON_DIR)
os.chdir(CATVTON_DIR)

from model.pipeline import CatVTONPipeline

BASE_CKPT = "booksforcharlie/stable-diffusion-inpainting"
ATTN_CKPT = "zhengchong/CatVTON"

catvton_pipe = CatVTONPipeline(
    base_ckpt=BASE_CKPT,
    attn_ckpt=ATTN_CKPT,
    attn_ckpt_version="vitonhd",
    weight_dtype=torch.float16,
    device=DEVICE,
    skip_safety_check=True,
    use_tf32=True,
)

print("✅ CatVTON pipeline loaded")

# ------------------------------------------------------------
# 4c. OpenCLIP (Q4 garment fidelity)
# ------------------------------------------------------------
import open_clip
import torch.nn.functional as F

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="laion2b_s34b_b79k"
)
clip_model = clip_model.to(DEVICE).eval()

print("✅ OpenCLIP loaded")

# ------------------------------------------------------------
# 4d. OpenCV face detector (Q4 identity preservation)
# ------------------------------------------------------------
face_cascade_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
face_detector = cv2.CascadeClassifier(face_cascade_path)
assert not face_detector.empty(), "Could not load OpenCV face detector."

print("✅ OpenCV face detector loaded")

# ------------------------------------------------------------
# 4e. MiniCPM-V-2_6-int4 (Q1 attributes + Q4 VLM judge)
# ------------------------------------------------------------
from transformers import AutoModel, AutoTokenizer

VLM_MODEL_ID = "openbmb/MiniCPM-V-2_6-int4"

print("Loading MiniCPM-V (first run downloads ~7GB)...")
vlm_model = AutoModel.from_pretrained(
    VLM_MODEL_ID, trust_remote_code=True, attn_implementation="sdpa",
)
vlm_model.eval()
vlm_tokenizer = AutoTokenizer.from_pretrained(VLM_MODEL_ID, trust_remote_code=True)

print("✅ MiniCPM-V loaded")

gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()
    print(f"\nGPU allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
    print(f"GPU reserved : {torch.cuda.memory_reserved()/1024**3:.2f} GB")

print("\n🎉 ALL MODELS READY")


Device: cuda
GPU: Tesla T4


Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

.gitattributes:   0%|          | 0.00/353 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

assets/demo_atr.png:   0%|          | 0.00/15.7k [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

✅ MiniCPM-V loaded

GPU allocated: 8.25 GB
GPU reserved : 8.40 GB

🎉 ALL MODELS READY


## 5. Q1 — VLM attribute extraction (garment attributes + person pose)

In [5]:
import json, re

MAX_EDGE = 1024

def load_and_resize(img: Image.Image) -> Image.Image:
    img = img.convert("RGB")
    w, h = img.size
    scale = MAX_EDGE / max(w, h)
    if scale < 1.0:
        img = img.resize((int(w * scale), int(h * scale)))
    return img

def _extract_json(text: str) -> dict:
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        raise ValueError(f"No JSON object found in response: {text[:200]}")
    return json.loads(match.group(0))


GARMENT_PROMPT = """You are a precise fashion attribute annotator. Look at the garment in the image
and output ONLY a single JSON object -- no prose, no markdown fences, no explanation.

Use exactly these keys:
{
  "type": "<e.g. t-shirt, blouse, jeans, dress, jacket, skirt, sweater, hoodie, shorts>",
  "sleeve_length": "<one of: sleeveless, short, 3/4, long, not_applicable, unknown>",
  "neckline": "<e.g. crew, v-neck, collared, scoop, turtleneck, halter, off-shoulder, not_applicable, unknown>",
  "primary_color": "<single dominant color, lowercase, 1-2 words max>",
  "pattern": "<e.g. solid, striped, graphic print, floral, plaid, polka dot, animal print, unknown>",
  "confidence_notes": "<short note on anything occluded, ambiguous, or uncertain; empty string if none>"
}

Now analyze the given garment image and return ONLY the JSON object."""


def get_garment_attributes(garment_img: Image.Image, max_retries: int = 1) -> dict:
    image = load_and_resize(garment_img)
    last_error = None
    for attempt in range(max_retries + 1):
        prompt = GARMENT_PROMPT
        if attempt > 0:
            prompt = GARMENT_PROMPT + f"\n\nYour previous response was invalid JSON or missing required keys ({last_error}). Fix it and return ONLY the corrected JSON object."
        msgs = [{"role": "user", "content": [image, prompt]}]
        response = vlm_model.chat(
            image=None, msgs=msgs, tokenizer=vlm_tokenizer,
            sampling=False, max_new_tokens=600
        )
        try:
            parsed = _extract_json(response)
            required = {"type", "sleeve_length", "neckline", "primary_color", "pattern"}
            if not required.issubset(parsed.keys()):
                raise ValueError(f"missing keys: {required - parsed.keys()}")
            return parsed
        except (ValueError, json.JSONDecodeError) as e:
            last_error = str(e)
            continue
    return {
        "type": "unknown", "sleeve_length": "unknown", "neckline": "unknown",
        "primary_color": "unknown", "pattern": "unknown",
        "confidence_notes": f"PARSE_FAILED after {max_retries + 1} attempts: {last_error}"
    }


PERSON_PROMPT = """You are a precise image annotator. Look at the photo of a person and return
ONLY a JSON object -- no prose, no markdown fences.

Work out each field in order, then use your earlier answers to fill in the later ones.

1. description: 1-2 sentences on what the person is doing, their orientation, and any occlusion.

2. torso_direction: which way the torso/chest is turned.
   - facing_camera: both shoulders and both sides of the chest are visible
   - facing_left / facing_right: one shoulder is toward camera, the other turned away or barely
     visible (includes strong three-quarter turns, not just full profile)
   - facing_away: back is to camera
   - unknown: cannot tell

3. head_direction: same options as torso_direction, but for the head/face. It can differ from
   torso_direction (e.g. torso turned sideways, face looking at camera) -- judge independently.

4. posture: standing, seated, or unknown.
   "seated" = sitting on any surface (chair, bench, floor, etc.), or knees visibly bent with
   hips at/below knee height.

5. visible_parts: list every body part visible anywhere in frame, even partially. Choose from:
   head, left_shoulder, right_shoulder, left_arm, right_arm, torso, hips, left_leg, right_leg,
   left_foot, right_foot.

6. pose_category: derive from posture and torso_direction, in this order:
   - posture is seated -> "seated" (overrides torso_direction no matter which way they face)
   - else torso_direction is facing_camera -> "front-facing"
   - else torso_direction is facing_left/right/away -> "side"
   - else -> "unknown"

7. upper_body_visible / lower_body_visible:
   - upper_body_visible = true if any meaningful upper-body region such as shoulders,
     arms, chest, or torso is visible
   - lower_body_visible = true if hips, legs, knees, or feet are visible

IMPORTANT:
- Classify pose primarily from body posture and torso orientation, not only face direction.
- A seated person remains "seated" even if facing directly toward the camera.
- Crossed arms alone do not mean the person is side-facing.
- If no person is visible, use:
  pose_category = "unknown",
  posture = "unknown",
  upper_body_visible = false,
  lower_body_visible = false.

Return exactly these keys, in this order:

{
  "description": "<1-2 sentences>",
  "torso_direction": "<facing_camera, facing_left, facing_right, facing_away, unknown>",
  "head_direction": "<facing_camera, facing_left, facing_right, facing_away, unknown>",
  "posture": "<standing, seated, unknown>",
  "visible_parts": ["<body parts>"],
  "pose_category": "<front-facing, side, seated, unknown>",
  "upper_body_visible": <true/false>,
  "lower_body_visible": <true/false>,
  "confidence_notes": "<short note on occlusion/ambiguity, or empty string>"
}

Now analyze the given person image and return ONLY the JSON object."""

SEATED_PROMPT = """Inspect ALL people visible in this image.

Determine whether at least one clearly visible person is SEATED.

A seated person has body weight supported by a chair, sofa,
bench, bed, floor, platform, or another surface.

Look for:
- supported hips
- bent knees
- sitting on a couch/chair/bench
- sitting posture even when torso is upright

Do NOT classify crossed arms alone as seated.

Return ONLY valid JSON:

{"seated_person_present": true}

or

{"seated_person_present": false}"""

ORIENTATION_PROMPT = """Analyze ONLY the MAIN PERSON'S BODY ORIENTATION.

Ignore face direction, gaze direction, arms/hands, and activity.
Focus ONLY on chest direction, torso rotation, shoulder alignment.

SIDE means the torso is rotated sideways, in a strong three-quarter pose, or one
shoulder/side is clearly more prominent than the other -- even if the face looks at camera.

FRONT-FACING means the chest is substantially square toward the camera and roughly symmetric.

Return ONLY:
{"body_orientation": "side"}
or
{"body_orientation": "front-facing"}"""


def get_person_attributes(person_img: Image.Image) -> dict:
    """Q1 person-attribute extraction: pose_category, upper/lower body visibility,
    plus a strong NO-PERSON guard used by the Q5 reject guardrail."""
    image = load_and_resize(person_img)

    msgs = [{"role": "user", "content": [image, PERSON_PROMPT]}]
    response = vlm_model.chat(
        image=None, msgs=msgs, tokenizer=vlm_tokenizer, sampling=False, max_new_tokens=500
    )

    try:
        parsed = _extract_json(response)
    except Exception as e:
        return {
            "description": "", "torso_direction": "unknown", "head_direction": "unknown",
            "posture": "unknown", "visible_parts": [], "pose_category": "unknown",
            "upper_body_visible": False, "lower_body_visible": False,
            "confidence_notes": f"PARSE_FAILED: {e}"
        }

    torso = str(parsed.get("torso_direction", "unknown")).strip().lower()
    posture = str(parsed.get("posture", "unknown")).strip().lower()
    upper_visible = parsed.get("upper_body_visible", False)
    lower_visible = parsed.get("lower_body_visible", False)
    confidence_notes = str(parsed.get("confidence_notes", "")).lower()

    no_person_detected = (
        (torso == "unknown" and posture == "unknown"
         and upper_visible is False and lower_visible is False)
        or "no person visible" in confidence_notes
        or "no person" in confidence_notes
    )

    if no_person_detected:
        parsed["pose_category"] = "unknown"
        parsed["upper_body_visible"] = False
        parsed["lower_body_visible"] = False
        return parsed

    # Focused seated re-check (seated has top priority)
    try:
        posture_msgs = [{"role": "user", "content": [image, SEATED_PROMPT]}]
        posture_response = vlm_model.chat(
            image=None, msgs=posture_msgs, tokenizer=vlm_tokenizer,
            sampling=False, max_new_tokens=150
        )
        try:
            posture_data = _extract_json(posture_response)
            seated_present = posture_data.get("seated_person_present", False) is True
        except Exception:
            text = posture_response.lower()
            positive = ["person is seated", "seated posture", "sitting on a couch", "sitting on a chair"]
            negative = ["not seated", "no seated person", "no person is seated", "there is no"]
            if any(p in text for p in negative):
                seated_present = False
            elif any(p in text for p in positive):
                seated_present = True
            else:
                seated_present = False
    except Exception:
        seated_present = False

    if seated_present:
        parsed["posture"] = "seated"
        parsed["pose_category"] = "seated"
        return parsed

    # Otherwise, trust structured torso_direction directly (no extra voting needed for a
    # single-image live demo -- keeps latency reasonable).
    if torso in {"facing_left", "facing_right", "facing_away"}:
        parsed["pose_category"] = "side"
    elif torso == "facing_camera":
        parsed["pose_category"] = "front-facing"
    else:
        parsed["pose_category"] = "unknown"

    return parsed

print("✅ Q1 functions ready: get_garment_attributes(), get_person_attributes()")


✅ Q1 functions ready: get_garment_attributes(), get_person_attributes()


## 6. Q2 — Human parsing, clothing mask, garment mask

In [6]:
from rembg import remove as rembg_remove

def run_human_parsing(person_img: Image.Image):
    """Run SCHP on the person image. Returns (parsing_map, is_person_present)."""
    image = person_img.convert("RGB")
    inputs = schp_processor(images=image, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = schp_model(**inputs)

    parsing_map = outputs.parsing_logits.argmax(dim=1)[0].cpu().numpy()
    prob = torch.softmax(outputs.parsing_logits[0], dim=0).cpu().numpy()

    unique_classes = np.unique(parsing_map)
    # Same "no person" logic used in Q2: only background predicted -> no person.
    is_person_present = len(unique_classes) > 1

    return parsing_map, prob, is_person_present


def get_upper_arm(arm_mask, fraction=0.6):
    """Keep only the top `fraction` of an arm mask (shoulder/upper-sleeve area)."""
    arm = (arm_mask > 0).astype(np.uint8)
    ys, xs = np.where(arm > 0)
    if len(ys) == 0:
        return np.zeros_like(arm)
    top, bottom = ys.min(), ys.max()
    height = max(bottom - top, 1)
    cutoff = int(top + fraction * height)
    upper = arm.copy()
    upper[cutoff + 1:, :] = 0
    return upper


def build_clothing_mask(person_img: Image.Image, prob: np.ndarray):
    """Q2-style clothing mask (upper clothes / dress / belt) + Q3-style upper-arm inclusion
    so CatVTON is free to change sleeve length. Generalised (no per-image manual tuning)."""
    W, H = person_img.size

    upper_prob = prob[4]   # upper clothes
    dress_prob = prob[7]   # dress
    belt_prob = prob[8]    # belt
    arm_l = prob[14]
    arm_r = prob[15]

    cloth_mask = ((upper_prob > 0.20) | (dress_prob > 0.20) | (belt_prob > 0.20)).astype(np.uint8)
    arm_l_mask = (arm_l > 0.20).astype(np.uint8)
    arm_r_mask = (arm_r > 0.20).astype(np.uint8)

    upper_arm_l = get_upper_arm(arm_l_mask, fraction=0.6)
    upper_arm_r = get_upper_arm(arm_r_mask, fraction=0.6)

    mask = np.maximum(cloth_mask, np.maximum(upper_arm_l, upper_arm_r))

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.dilate(mask, kernel, iterations=1)

    mask = cv2.resize(mask, (W, H), interpolation=cv2.INTER_NEAREST)
    return mask  # uint8, {0,1}


def get_garment_mask(garment_img: Image.Image) -> Image.Image:
    """rembg background removal -> alpha channel is the garment mask (Q2)."""
    rgba_img = garment_img.convert("RGBA")
    result = rembg_remove(rgba_img)
    rgba = np.array(result)
    mask = rgba[:, :, 3]
    return Image.fromarray(mask)

print("✅ Q2 functions ready: run_human_parsing(), build_clothing_mask(), get_garment_mask()")


✅ Q2 functions ready: run_human_parsing(), build_clothing_mask(), get_garment_mask()


## 7. Guardrails

In [7]:
def check_no_person(person_img: Image.Image, parsing_present: bool, person_attrs: dict):
    """
    REJECT guardrail. Uses two independent signals for robustness:
      1. SCHP parsing map -- only background class predicted (Q2 logic)
      2. Q1 VLM no-person guard -- pose_category == "unknown" AND both
         upper/lower body visibility are False
    If either signal says "no person", the image is rejected.
    """
    vlm_no_person = (
        person_attrs.get("pose_category") == "unknown"
        and person_attrs.get("upper_body_visible") is False
        and person_attrs.get("lower_body_visible") is False
    )
    no_person = (not parsing_present) or vlm_no_person
    return no_person


def check_pose_warning(person_attrs: dict):
    """
    WARN guardrail (does not block processing). Fires for seated or side poses,
    which are known to produce less reliable garment-fit geometry.
    """
    pose = person_attrs.get("pose_category", "unknown")
    if pose == "seated":
        return True, "⚠️ Seated pose detected — garment fit around the hips/lap may be less reliable."
    if pose == "side":
        return True, "⚠️ Side / turned pose detected — garment fit on the far side of the body may be less reliable."
    return False, ""


def estimate_processing_time(person_attrs=None):
    """
    Static per-stage time budget on a T4 GPU, used to show the user an estimate
    BEFORE the run starts. Q1 can call the VLM up to 2 extra times for
    seated/side disambiguation, so the estimate widens for non-frontal poses.
    """
    vlm_calls_base = 2       # 1x person pass + 1x garment pass
    vlm_calls_extra = 1       # seated re-check pass (Q1 always makes this call)
    schp_time = 3
    rembg_time = 3
    catvton_time = 45         # 50 steps @ 768x1024 on T4
    quality_time = 15         # CLIP + SSIM + 1 VLM judge call

    vlm_time_per_call = 8
    total_low = (vlm_calls_base * vlm_time_per_call) + schp_time + rembg_time + catvton_time + quality_time
    total_high = total_low + (vlm_calls_extra * vlm_time_per_call) + 20  # slack for orientation ambiguity

    return total_low, total_high

print("✅ Guardrail functions ready: check_no_person(), check_pose_warning(), estimate_processing_time()")


✅ Guardrail functions ready: check_no_person(), check_pose_warning(), estimate_processing_time()


## 8. Q3 — CatVTON try-on inference

In [18]:
import time, os

CATVTON_WIDTH, CATVTON_HEIGHT = 768, 1024
Q3_RESULTS_DIR = os.path.join(PROJECT_DIR, "Q3", "results")

# Exact filenames sitting in Q3/results/ (confirmed from os.listdir)
KNOWN_RESULT_FILES = {
    "01": "pair_01_tryon.png",
    "02": "pair_02_result.png",
    "03": "pair_03_catvton_result.png",
    "04": "pair_04_result.png",
    "05": "pair_05_catvton_result.png",
}

def find_existing_result(person_path: str, garment_path: str):
    """Look for a pre-generated Q3 try-on result matching this person/garment pair."""
    person_stem = os.path.splitext(os.path.basename(person_path))[0]
    garment_stem = os.path.splitext(os.path.basename(garment_path))[0]

    pair_num = None
    for stem in (person_stem, garment_stem):
        digits = ''.join(ch for ch in stem if ch.isdigit())
        if digits:
            pair_num = digits.zfill(2)
            break

    if pair_num and pair_num in KNOWN_RESULT_FILES:
        path = os.path.join(Q3_RESULTS_DIR, KNOWN_RESULT_FILES[pair_num])
        if os.path.isfile(path):
            return path
    return None


def run_catvton(person_img: Image.Image, garment_img: Image.Image, mask_np: np.ndarray,
                 steps: int = 50, cfg: float = 2.5, seed: int = 42,
                 person_path: str = None, garment_path: str = None):
    """
    If a pre-generated Q3 result exists for this person/garment pair, reuse it.
    Otherwise run CatVTON inference fresh.
    mask_np is a uint8 {0,1} array in person-image resolution.
    """
    person_rgb = person_img.convert("RGB")
    garment_rgb = garment_img.convert("RGB")
    mask_img = Image.fromarray((mask_np * 255).astype(np.uint8)).convert("L")

    if person_path and garment_path:
        existing = find_existing_result(person_path, garment_path)
        if existing:
            print(f"✅ Using pre-generated Q3 result: {existing}")
            result_img = Image.open(existing).convert("RGB").resize(person_rgb.size, Image.Resampling.LANCZOS)
            return result_img, mask_img

    person_in = person_rgb.resize((CATVTON_WIDTH, CATVTON_HEIGHT), Image.Resampling.LANCZOS)
    garment_in = garment_rgb.resize((CATVTON_WIDTH, CATVTON_HEIGHT), Image.Resampling.LANCZOS)
    mask_in = mask_img.resize((CATVTON_WIDTH, CATVTON_HEIGHT), Image.Resampling.NEAREST)
    mask_in = mask_in.point(lambda p: 255 if p > 127 else 0)

    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    generator = torch.Generator(device=DEVICE).manual_seed(seed)

    with torch.inference_mode():
        result = catvton_pipe(
            image=person_in,
            condition_image=garment_in,
            mask=mask_in,
            num_inference_steps=steps,
            guidance_scale=cfg,
            generator=generator,
        )

    if isinstance(result, (list, tuple)):
        result_img = result[0]
    elif hasattr(result, "images"):
        result_img = result.images[0]
    else:
        result_img = result

    if not isinstance(result_img, Image.Image):
        result_img = Image.fromarray(result_img)

    result_img = result_img.convert("RGB").resize(person_rgb.size, Image.Resampling.LANCZOS)
    return result_img, mask_img

print("✅ Q3 function ready: run_catvton()")

✅ Q3 function ready: run_catvton()


## 9. Q4 — Automated quality evaluation

In [9]:
JUDGE_RUBRIC = """
You are an evaluator of virtual try-on image quality.

You are given three images in this exact order:

IMAGE 1: Original person
IMAGE 2: Target garment
IMAGE 3: Generated virtual try-on result

Evaluate IMAGE 3 by comparing it with IMAGE 1 and IMAGE 2.

Evaluate these three criteria:

1. FIT REALISM
Assess whether the target garment fits naturally on the person's body.
Check the neckline, shoulders, sleeves, torso, garment boundaries,
body pose, and overall geometric plausibility.

2. ARTIFACTS
Look for visible generation problems such as distorted anatomy,
warped arms, broken garment edges, duplicated fabric, unnatural
blending, missing regions, incorrect boundaries, or other artifacts.
Fewer and less severe artifacts should result in a higher score.

3. TEXTURE TRANSFER
Compare IMAGE 3 with the target garment in IMAGE 2.
Check whether the garment's color, pattern, print, texture, and
important visual details have been preserved.

Give ONE overall quality score from 1 to 10.

Scoring guide:

1-2: Failed try-on with severe errors or incorrect garment.
3-4: Poor result with major fit, artifact, or texture problems.
5-6: Acceptable result but with clearly noticeable problems.
7-8: Good result with realistic transfer and only minor imperfections.
9: Excellent result with very small defects.
10: Exceptional, near-perfect virtual try-on.

Return ONLY valid JSON using exactly this structure:

{
  "score": 8,
  "reason": "A concise explanation discussing fit realism, artifacts, and texture transfer."
}

Requirements:
- score must be an integer from 1 to 10
- reason must discuss the visible result
- do not output markdown
- do not output text outside the JSON
"""


@torch.no_grad()
def clip_embedding(image: Image.Image):
    tensor = clip_preprocess(image).unsqueeze(0).to(DEVICE)
    features = clip_model.encode_image(tensor)
    return F.normalize(features.float(), dim=-1)


@torch.no_grad()
def garment_fidelity_score(garment_img: Image.Image, result_img: Image.Image, mask_np: np.ndarray):
    """OpenCLIP cosine similarity between the target garment and the generated garment region."""
    result_np = np.array(result_img.convert("RGB"))
    mask_bool = mask_np.astype(bool)
    if not np.any(mask_bool):
        return None

    ys, xs = np.where(mask_bool)
    x1, x2, y1, y2 = xs.min(), xs.max(), ys.min(), ys.max()
    crop = result_np[y1:y2 + 1, x1:x2 + 1]
    crop_mask = mask_bool[y1:y2 + 1, x1:x2 + 1]

    neutral = np.full_like(crop, 255)
    garment_region = np.where(crop_mask[..., None], crop, neutral)
    garment_region_img = Image.fromarray(garment_region.astype(np.uint8))

    garment_emb = clip_embedding(garment_img.convert("RGB"))
    region_emb = clip_embedding(garment_region_img)

    return float(F.cosine_similarity(garment_emb, region_emb).item())


def extract_face_bbox(img: Image.Image, padding: float = 0.20):
    """Detect the largest face using OpenCV Haar cascade. Returns bbox or None."""
    image_bgr = cv2.cvtColor(np.array(img.convert("RGB")), cv2.COLOR_RGB2BGR)
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    faces = face_detector.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(50, 50))
    if len(faces) == 0:
        return None
    x, y, w, h = max(faces, key=lambda box: box[2] * box[3])
    pad_x, pad_y = int(w * padding), int(h * padding)
    x1 = max(0, x - pad_x)
    y1 = max(0, y - pad_y)
    x2 = min(image_bgr.shape[1], x + w + pad_x)
    y2 = min(image_bgr.shape[0], y + h + pad_y)
    return (x1, y1, x2, y2)


def identity_preservation_score(person_img: Image.Image, result_img: Image.Image):
    """
    Face-region SSIM between original and generated image, using the SAME bbox
    (detected on the original) for both crops. Returns (score, bbox) or (None, None)
    if no face could be detected (common for seated/side/no-person edge cases).
    """
    from skimage.metrics import structural_similarity as ssim_fn

    original = cv2.cvtColor(np.array(person_img.convert("RGB")), cv2.COLOR_RGB2BGR)
    generated = cv2.cvtColor(np.array(result_img.convert("RGB")), cv2.COLOR_RGB2BGR)

    if generated.shape[:2] != original.shape[:2]:
        generated = cv2.resize(generated, (original.shape[1], original.shape[0]), interpolation=cv2.INTER_AREA)

    bbox = extract_face_bbox(person_img, padding=0.20)
    if bbox is None:
        return None, None

    x1, y1, x2, y2 = bbox
    original_face = original[y1:y2, x1:x2]
    generated_face = generated[y1:y2, x1:x2]
    if original_face.size == 0 or generated_face.size == 0:
        return None, None

    FACE_SIZE = (256, 256)
    original_face = cv2.resize(original_face, FACE_SIZE, interpolation=cv2.INTER_AREA)
    generated_face = cv2.resize(generated_face, FACE_SIZE, interpolation=cv2.INTER_AREA)

    original_gray = cv2.cvtColor(original_face, cv2.COLOR_BGR2GRAY)
    generated_gray = cv2.cvtColor(generated_face, cv2.COLOR_BGR2GRAY)

    score = ssim_fn(original_gray, generated_gray, data_range=255)
    return float(score), bbox


def parse_judge_response(response):
    if not isinstance(response, str):
        response = str(response)
    text = response.strip().replace("```json", "").replace("```", "").strip()
    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*?\}", text, flags=re.DOTALL)
        if not match:
            raise ValueError(f"Could not find JSON in VLM response:\n{text}")
        data = json.loads(match.group(0))
    score = int(data["score"])
    reason = str(data["reason"]).strip()
    if not 1 <= score <= 10:
        raise ValueError(f"Invalid VLM score: {score}")
    return score, reason


def vlm_judge(person_img: Image.Image, garment_img: Image.Image, result_img: Image.Image):
    """VLM-as-judge (same MiniCPM-V model as Q1) scoring fit realism / artifacts / texture transfer."""
    msgs = [{
        "role": "user",
        "content": [person_img.convert("RGB"), garment_img.convert("RGB"),
                    result_img.convert("RGB"), JUDGE_RUBRIC]
    }]
    response = vlm_model.chat(
        image=None, msgs=msgs, tokenizer=vlm_tokenizer, sampling=False, max_new_tokens=350
    )
    try:
        score, reason = parse_judge_response(response)
    except Exception as e:
        score, reason = None, f"VLM judge parse failed: {e}"
    return score, reason

print("✅ Q4 functions ready: garment_fidelity_score(), identity_preservation_score(), vlm_judge()")


✅ Q4 functions ready: garment_fidelity_score(), identity_preservation_score(), vlm_judge()


## 10. Main orchestrator — full Q1→Q4 pipeline with guardrails

In [19]:
def process_tryon(person_img: Image.Image, garment_img: Image.Image, progress_cb=None,
                   person_path: str = None, garment_path: str = None):
    """
    Runs the full Q1 -> Q2 -> Q3 -> Q4 pipeline with guardrails.

    person_path / garment_path are OPTIONAL. If provided, run_catvton() first checks
    Q3/results/ for a pre-generated try-on matching that pair and reuses it instead of
    re-running CatVTON. Uploads with no path always run inference fresh.
    """
    def log(msg):
        print(msg)
        if progress_cb:
            progress_cb(msg)

    t_start = time.time()
    est_low, est_high = estimate_processing_time()

    out = {
        "status": "ok",
        "reject_reason": None,
        "warnings": [],
        "result_image": None,
        "garment_attributes": None,
        "person_attributes": None,
        "quality_scores": None,
        "elapsed_seconds": None,
        "estimated_range": (est_low, est_high),
    }

    if person_img is None or garment_img is None:
        out["status"] = "rejected"
        out["reject_reason"] = "Please upload both a person image and a garment image."
        out["elapsed_seconds"] = round(time.time() - t_start, 1)
        return out

    person_img = person_img.convert("RGB")
    garment_img = garment_img.convert("RGB")

    log("Running human parsing (Q2 / SCHP)...")
    parsing_map, prob, parsing_person_present = run_human_parsing(person_img)

    log("Running VLM person analysis (Q1)...")
    person_attrs = get_person_attributes(person_img)
    out["person_attributes"] = person_attrs

    if check_no_person(person_img, parsing_person_present, person_attrs):
        out["status"] = "rejected"
        out["reject_reason"] = (
            "❌ No person detected in the uploaded image. "
            "Please upload a clear photo containing a visible person."
        )
        out["elapsed_seconds"] = round(time.time() - t_start, 1)
        log(out["reject_reason"])
        return out

    warn, warn_msg = check_pose_warning(person_attrs)
    if warn:
        out["warnings"].append(warn_msg)
        log(warn_msg)

    log("Running VLM garment analysis (Q1)...")
    garment_attrs = get_garment_attributes(garment_img)
    out["garment_attributes"] = garment_attrs

    log("Building clothing mask (Q2)...")
    clothing_mask = build_clothing_mask(person_img, prob)

    log("Getting try-on result (Q3)...")
    result_img, mask_img = run_catvton(
        person_img, garment_img, clothing_mask,
        person_path=person_path, garment_path=garment_path,
    )
    out["result_image"] = result_img

    log("Computing quality scores (Q4)...")
    mask_np_full = (np.array(mask_img.resize(person_img.size, Image.Resampling.NEAREST)) > 127).astype(np.uint8)

    fidelity = garment_fidelity_score(garment_img, result_img, mask_np_full)
    identity, face_bbox = identity_preservation_score(person_img, result_img)
    vlm_score, vlm_reason = vlm_judge(person_img, garment_img, result_img)

    if identity is None:
        out["warnings"].append(
            "⚠️ No face detected for identity-preservation scoring "
            "(expected for side/seated poses) -- identity_preservation_score is N/A."
        )

    out["quality_scores"] = {
        "garment_fidelity_score": round(fidelity, 4) if fidelity is not None else None,
        "identity_preservation_score": round(identity, 4) if identity is not None else None,
        "vlm_judge_score": vlm_score,
        "vlm_judge_reason": vlm_reason,
    }

    out["elapsed_seconds"] = round(time.time() - t_start, 1)
    log(f"✅ Done in {out['elapsed_seconds']}s")
    return out

print("✅ Orchestrator ready: process_tryon(person_img, garment_img, person_path=None, garment_path=None)")

✅ Orchestrator ready: process_tryon(person_img, garment_img, person_path=None, garment_path=None)


## 11. Gradio web app

In [20]:
import gradio as gr

KNOWN_PAIRS = {
    f"pair_{i:02d}": {
        "person": os.path.join(PROJECT_DIR, "Q1", "person", f"person_{i:02d}.png"),
        "garment": os.path.join(PROJECT_DIR, "Q1", "garment", f"garment_{i:02d}.jpg"),
    }
    for i in range(1, 6)
}

def format_person_attrs(attrs):
    if not attrs:
        return "-"
    keys = ["pose_category", "posture", "torso_direction", "upper_body_visible", "lower_body_visible"]
    return "\n".join(f"**{k}**: {attrs.get(k)}" for k in keys if k in attrs)

def format_garment_attrs(attrs):
    if not attrs:
        return "-"
    keys = ["type", "sleeve_length", "neckline", "primary_color", "pattern"]
    return "\n".join(f"**{k}**: {attrs.get(k)}" for k in keys if k in attrs)

def format_scores(scores):
    if not scores:
        return "-"
    lines = [
        f"**Garment fidelity (OpenCLIP cosine sim)**: {scores['garment_fidelity_score']}",
        f"**Identity preservation (face SSIM)**: {scores['identity_preservation_score']}",
        f"**VLM judge score (1-10)**: {scores['vlm_judge_score']}",
        f"**VLM judge reason**: {scores['vlm_judge_reason']}",
    ]
    return "\n\n".join(lines)


def gradio_run(person_img, garment_img):
    """Freeform upload path -- no filenames available, always runs CatVTON fresh."""
    est_low, est_high = estimate_processing_time()
    result = process_tryon(person_img, garment_img)

    if result["status"] == "rejected":
        banner = f"### {result['reject_reason']}"
        return (banner, None, "-", "-", "-",
                f"Elapsed: {result['elapsed_seconds']}s (rejected before generation)")

    banner_lines = ["### ✅ Try-on generated"]
    for w in result["warnings"]:
        banner_lines.append(w)
    banner = "\n\n".join(banner_lines)

    return (
        banner,
        result["result_image"],
        format_garment_attrs(result["garment_attributes"]),
        format_person_attrs(result["person_attributes"]),
        format_scores(result["quality_scores"]),
        f"Elapsed: {result['elapsed_seconds']}s  (estimated range was {est_low}-{est_high}s)",
    )


def gradio_run_known_pair(pair_id):
    """Known-pair path -- has filenames, so it reuses the matching Q3/results/ image."""
    if not pair_id:
        return "### Please select a pair.", None, "-", "-", "-", "-"

    p_path = KNOWN_PAIRS[pair_id]["person"]
    g_path = KNOWN_PAIRS[pair_id]["garment"]
    person_img = Image.open(p_path)
    garment_img = Image.open(g_path)

    result = process_tryon(person_img, garment_img, person_path=p_path, garment_path=g_path)

    if result["status"] == "rejected":
        return f"### {result['reject_reason']}", None, "-", "-", "-", "-"

    banner_lines = ["### ✅ Try-on shown (reused from Q3/results/ if available)"]
    for w in result["warnings"]:
        banner_lines.append(w)
    banner = "\n\n".join(banner_lines)

    return (
        banner,
        result["result_image"],
        format_garment_attrs(result["garment_attributes"]),
        format_person_attrs(result["person_attributes"]),
        format_scores(result["quality_scores"]),
        f"Elapsed: {result['elapsed_seconds']}s",
    )


with gr.Blocks(title="Q5 - Mini Virtual Try-On Demo") as demo:
    gr.Markdown("# 👕 Mini Virtual Try-On Demo\n"
                "Upload a **person** photo and a **garment** photo, or pick one of the "
                "5 known Q1 pairs below to instantly reuse its pre-generated Q3 result.")

    with gr.Row():
        person_in = gr.Image(type="pil", label="Person image (upload)")
        garment_in = gr.Image(type="pil", label="Garment image (upload)")

    run_btn = gr.Button("Run Try-On (new upload)", variant="primary")

    with gr.Row():
        pair_dropdown = gr.Dropdown(choices=list(KNOWN_PAIRS.keys()), label="Or pick a known pair (reuses Q3/results/)")
        load_pair_btn = gr.Button("Load Pair")

    status_box = gr.Markdown()

    with gr.Row():
        result_out = gr.Image(type="pil", label="Try-on result")
        with gr.Column():
            garment_attrs_out = gr.Markdown(label="Q1 garment attributes")
            person_attrs_out = gr.Markdown(label="Q1 person attributes")
            scores_out = gr.Markdown(label="Q4 quality scores")

    time_out = gr.Markdown()

    run_btn.click(
        fn=gradio_run,
        inputs=[person_in, garment_in],
        outputs=[status_box, result_out, garment_attrs_out, person_attrs_out, scores_out, time_out],
    )

    load_pair_btn.click(
        fn=gradio_run_known_pair,
        inputs=[pair_dropdown],
        outputs=[status_box, result_out, garment_attrs_out, person_attrs_out, scores_out, time_out],
    )

    gr.Markdown(
        "### Guardrail test images (for the demo video)\n"
        "Upload each of these (with any garment) to see the guardrails fire:\n"
        "- `edge_cases/no_person.jpg` -> **rejected**\n"
        "- `edge_cases/person_seated.jpg` -> **warning**\n"
        "- `edge_cases/person_side_pose.jpg` -> **warning**"
    )

demo.queue()
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2f63f1344ea4f5fe18.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1368: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Running human parsing (Q2 / SCHP)...
Running VLM person analysis (Q1)...


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1368: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Running human parsing (Q2 / SCHP)...
Running VLM person analysis (Q1)...
Running VLM garment analysis (Q1)...
Running VLM garment analysis (Q1)...
Building clothing mask (Q2)...
Getting try-on result (Q3)...


...

Computing quality scores (Q4)...
✅ Done in 246.9s


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1368: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Running human parsing (Q2 / SCHP)...
Running VLM person analysis (Q1)...
Running VLM garment analysis (Q1)...
Building clothing mask (Q2)...
Getting try-on result (Q3)...


100%|██████████| 50/50 [01:40<00:00,  2.02s/it]


Computing quality scores (Q4)...
✅ Done in 243.9s
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://2f63f1344ea4f5fe18.gradio.live


In [17]:
import os
Q3_RESULTS_DIR = os.path.join(PROJECT_DIR, "Q3", "results")
print(os.listdir(Q3_RESULTS_DIR))

['pair_01_tryon.png', 'pair_02_result.png', 'pair_04_result.png', 'pair_05_catvton_result.png', 'pair_03_catvton_result.png']


## 12. Standalone guardrail test (for the demo video)

Runs `process_tryon()` directly on the three required edge-case images so you can capture
the guardrail behaviour without needing to click through the UI. This searches a few likely
Drive locations for the files (matching where Q1/Q2 stored their `edge_cases/` folders) —
adjust `EDGE_CANDIDATES` if your files live elsewhere.

In [21]:
def find_edge_case(filename):
    candidates = [
        os.path.join(PROJECT_DIR, "Q1", "edge_cases", filename),
        os.path.join(PROJECT_DIR, "Q2", "dataset", "sample_files", "edge_cases", filename),
        os.path.join(PROJECT_DIR, "edge_cases", filename),
    ]
    for path in candidates:
        if os.path.isfile(path):
            return path
    raise FileNotFoundError(f"Could not find {filename} in any of: {candidates}")


# Use any garment for these guardrail tests -- the reject/warning fires on the PERSON image.
sample_garment_path = os.path.join(PROJECT_DIR, "Q1", "garment", "garment_01.jpg")
sample_garment = Image.open(sample_garment_path).convert("RGB")

edge_case_files = {
    "no_person.jpg": "expected: REJECTED",
    "person_seated.jpg": "expected: WARNING (seated)",
    "person_side_pose.jpg": "expected: WARNING (side)",
}

for filename, expectation in edge_case_files.items():
    print("=" * 70)
    print(f"TESTING: {filename}   ({expectation})")
    print("=" * 70)

    path = find_edge_case(filename)
    person_img = Image.open(path).convert("RGB")

    result = process_tryon(person_img, sample_garment)

    print("Status  :", result["status"])
    if result["status"] == "rejected":
        print("Reason  :", result["reject_reason"])
    else:
        print("Warnings:", result["warnings"] if result["warnings"] else "(none)")
        print("Pose    :", result["person_attributes"].get("pose_category"))
    print("Elapsed :", result["elapsed_seconds"], "s")
    print()

print("🎉 Guardrail test complete -- capture this cell's output (and/or the Gradio UI runs) for the demo video.")


TESTING: no_person.jpg   (expected: REJECTED)
Running human parsing (Q2 / SCHP)...
Running VLM person analysis (Q1)...
❌ No person detected in the uploaded image. Please upload a clear photo containing a visible person.
Status  : rejected
Reason  : ❌ No person detected in the uploaded image. Please upload a clear photo containing a visible person.
Elapsed : 36.8 s

TESTING: person_seated.jpg   (expected: WARNING (seated))
Running human parsing (Q2 / SCHP)...
Running VLM person analysis (Q1)...
⚠️ Seated pose detected — garment fit around the hips/lap may be less reliable.
Running VLM garment analysis (Q1)...
Building clothing mask (Q2)...
Getting try-on result (Q3)...


100%|██████████| 50/50 [01:40<00:00,  2.01s/it]


Computing quality scores (Q4)...
✅ Done in 245.7s
Status  : ok
Warnings: ['⚠️ Seated pose detected — garment fit around the hips/lap may be less reliable.']
Pose    : seated
Elapsed : 245.7 s

TESTING: person_side_pose.jpg   (expected: WARNING (side))
Running human parsing (Q2 / SCHP)...
Running VLM person analysis (Q1)...
Running VLM garment analysis (Q1)...
Building clothing mask (Q2)...
Getting try-on result (Q3)...


100%|██████████| 50/50 [01:40<00:00,  2.01s/it]


Computing quality scores (Q4)...
✅ Done in 237.7s
Status  : ok
Warnings: (none)
Pose    : front-facing
Elapsed : 237.7 s

🎉 Guardrail test complete -- capture this cell's output (and/or the Gradio UI runs) for the demo video.
